# One tool, many ID formats: let the tool detect the type

When you give Claude a tool that looks something like `lookup_status(id)`, you eventually
hit a system where the same conceptual entity is addressed by **more than one ID format**.
A payments platform is the classic example: a single "transaction" might be referred to by
an *order ID*, a *payment ID*, or a *checkout-session ID*, each living behind a different
backend service with its own ID shape.

How should the tool be designed so Claude uses it correctly?

This notebook walks through three options and shows why, when the ID formats are structurally
distinct, the cleanest design is to let the **tool** detect the ID type rather than asking
the **model** to declare it. The lesson generalizes to any API where one entity spans several
ID spaces (orders vs. payments vs. sessions vs. receipts).

## What you'll learn

- Why exposing one tool per ID space inflates the model's decision surface
- Why a `id_type` parameter pushes work onto the model that the tool can do deterministically
- How to route by ID structure inside the tool, so the caller passes any ID and it just works
- When this pattern does **not** apply (ambiguous/overlapping ID formats)

## Setup

In [ ]:
%pip install -qU anthropic

In [2]:
import os
import re

from anthropic import Anthropic

client = Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

# Latest Haiku — cheap and more than capable for a single-tool demo.
MODEL = "claude-haiku-4-5"

## The scenario

We have one conceptual entity — a *transaction* — addressed through three different ID spaces,
each owned by a different backend. The ID formats are structurally distinct:

| Backend | ID format | Example |
|---|---|---|
| Checkout sessions | `<hex>-<hex>` | `a1b2c3d4-9f8e7d6c` |
| Orders | starts with `ORD-` | `ORD-20260115-7741` |
| Payments | digits only | `4051887766` |

Below are three mock backends standing in for those services. In a real integration these
would be HTTP calls to three different endpoints.

In [3]:
# Each backend addresses the same conceptual entity through a different ID space.
CHECKOUT_SESSIONS = {
    "a1b2c3d4-9f8e7d6c": {"state": "expired", "amount": 4990, "method": "card"},
}
ORDERS = {
    "ORD-20260115-7741": {"state": "paid", "amount": 12900, "method": "pix"},
}
PAYMENTS = {
    "4051887766": {"state": "refunded", "amount": 4990, "method": "card"},
}

## Approach A: one tool per ID space

The most literal design mirrors the backends: three tools, one per service.

```python
tools = [
    {"name": "get_checkout_session", ...},
    {"name": "get_order", ...},
    {"name": "get_payment", ...},
]
```

This works, but it has costs:

- **Bigger decision surface.** Every extra tool is another choice Claude has to get right on
  each turn. Three near-identical tools that differ only by ID space invite mis-selection.
- **The model must know the mapping.** To pick `get_order` for `ORD-20260115-7741`, the model
  has to have internalized that `ORD-` means "order". You're teaching ID conventions through
  tool names.
- **It doesn't scale.** A fourth ID space (refunds, receipts, …) means a fourth tool.

The split exists in your backend. That doesn't mean it has to exist in the model's view of
the world.

## Approach B: one tool, but the model declares the type

A tempting middle ground is a single tool with an explicit `id_type` parameter:

```python
{
    "name": "lookup_status",
    "input_schema": {
        "type": "object",
        "properties": {
            "id": {"type": "string"},
            "id_type": {"enum": ["order", "payment", "checkout_session"]},
        },
        "required": ["id", "id_type"],
    },
}
```

This collapses three tools into one, which is good. But it pushes a **deterministic
classification** onto the model: given `ORD-20260115-7741`, set `id_type="order"`. The model
can do this, but:

- It's redundant — the ID already carries its type in its structure.
- It's a place to be wrong — the model can mislabel, and now the tool faithfully queries the
  wrong backend.
- It adds a required field the caller (and the model) have to reason about every time.

Anything the tool can determine from the input itself is work the model shouldn't have to do.

## Approach C: one tool, the tool detects the type

The IDs are structurally distinct, so the tool can route by shape alone. The caller — human
or model — passes whatever ID it has, and the tool figures out which backend owns it.

In [4]:
def lookup_status(transaction_id: str) -> dict:
    """Look up a transaction by ID, routing to the backend that owns its ID space.

    The three ID formats are structurally distinct, so the tool can detect the type
    from the ID itself. The caller never has to know which backend an ID belongs to.
    """
    if transaction_id.startswith("ORD-"):
        record, source = ORDERS.get(transaction_id), "orders"
    elif transaction_id.isdigit():
        record, source = PAYMENTS.get(transaction_id), "payments"
    elif re.fullmatch(r"[0-9a-f]+-[0-9a-f]+", transaction_id):
        record, source = CHECKOUT_SESSIONS.get(transaction_id), "checkout_sessions"
    else:
        return {"error": f"Unrecognized ID format: {transaction_id!r}"}

    if record is None:
        return {"error": f"No {source} record for {transaction_id!r}"}
    return {"source": source, **record}

The routing is deterministic and testable on its own, with no model in the loop:

In [5]:
for tid in [
    "ORD-20260115-7741",  # order
    "4051887766",  # payment
    "a1b2c3d4-9f8e7d6c",  # checkout session
    "not-a-real-id!",  # unrecognized
]:
    print(f"{tid:<20} -> {lookup_status(tid)}")

ORD-20260115-7741    -> {'source': 'orders', 'state': 'paid', 'amount': 12900, 'method': 'pix'}
4051887766           -> {'source': 'payments', 'state': 'refunded', 'amount': 4990, 'method': 'card'}
a1b2c3d4-9f8e7d6c    -> {'source': 'checkout_sessions', 'state': 'expired', 'amount': 4990, 'method': 'card'}
not-a-real-id!       -> {'error': "Unrecognized ID format: 'not-a-real-id!'"}


## Wiring it to Claude

The tool Claude sees has a **single** `transaction_id` parameter. No `id_type`, no choice
between near-identical tools. Note how the description tells Claude it can pass any format and
to pass the ID verbatim — the routing is the tool's job, not Claude's.

In [6]:
tools = [
    {
        "name": "lookup_status",
        "description": (
            "Look up the current status of a transaction by its ID. Accepts any of the "
            "platform's ID formats (order IDs, payment IDs, or checkout-session IDs) and "
            "resolves the correct backend automatically. Pass the ID exactly as the user "
            "gave it; do not try to classify or reformat it."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "transaction_id": {
                    "type": "string",
                    "description": "The transaction ID, in any of the platform's formats.",
                }
            },
            "required": ["transaction_id"],
        },
    }
]

In [7]:
def run_conversation(user_message: str) -> str:
    """Run a single tool-use round-trip and return Claude's final text answer."""
    messages = [{"role": "user", "content": user_message}]

    response = client.messages.create(model=MODEL, max_tokens=1024, tools=tools, messages=messages)

    # Resolve tool calls until Claude stops requesting them.
    while response.stop_reason == "tool_use":
        messages.append({"role": "assistant", "content": response.content})

        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                result = lookup_status(**block.input)
                print(f"  [tool] lookup_status({block.input}) -> {result}")
                tool_results.append(
                    {
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": str(result),
                    }
                )

        messages.append({"role": "user", "content": tool_results})
        response = client.messages.create(
            model=MODEL, max_tokens=1024, tools=tools, messages=messages
        )

    return "".join(block.text for block in response.content if block.type == "text")

Ask about two transactions in different ID spaces in a single turn:

In [8]:
answer = run_conversation(
    "Can you check on two things for me? Order ORD-20260115-7741, and payment 4051887766."
)
print("\n" + answer)

  [tool] lookup_status({'transaction_id': 'ORD-20260115-7741'}) -> {'source': 'orders', 'state': 'paid', 'amount': 12900, 'method': 'pix'}
  [tool] lookup_status({'transaction_id': '4051887766'}) -> {'source': 'payments', 'state': 'refunded', 'amount': 4990, 'method': 'card'}



Here's the status for both transactions:

**Order ORD-20260115-7741:**
- Status: Paid
- Amount: 12,900
- Payment Method: PIX

**Payment 4051887766:**
- Status: Refunded
- Amount: 4,990
- Payment Method: Card

Is there anything else you'd like to know about these transactions?


Claude calls `lookup_status` once per ID, passing each one verbatim. It never had to know that
`ORD-` means "order" or that a bare number is a payment — the tool resolved both. Adding a
fourth ID space later is a change to the tool, not to the tool *contract* Claude reasons about.

## When NOT to use this

Auto-detection works because the ID formats here are **mutually exclusive** — no string is a
valid order ID *and* a valid payment ID. The pattern breaks down when:

- **Formats overlap.** If two ID spaces can both be "digits only", structure no longer
  disambiguates them. Fall back to a type parameter (Approach B) or separate tools (Approach A).
- **Detection is expensive or unreliable.** If telling the types apart requires a network
  round-trip or a fuzzy heuristic, an explicit `id_type` is clearer and cheaper.
- **The types are genuinely different operations.** If "look up an order" and "look up a
  payment" return different shapes the caller treats differently, separate tools may model the
  domain better.

The test is simple: *can the receiver determine the type from the input alone, cheaply and
unambiguously?* If yes, do it in the tool. If no, make the type explicit.

## Takeaways

- Every tool and every required parameter is decision surface for the model. Removing surface
  the model doesn't need makes tool use more reliable.
- When an input carries its own type signature, let the **receiver** detect the type. Don't
  push deterministic classification onto the model.
- This keeps one clean capability (`lookup_status(id)`) stable even as the number of backend
  ID spaces grows.